## Model Context Protocol

Model context protocol (MCP) is a communicaiton layer that provides Claude with context and tools without requiring you to write a bunch of tedious integration code. Think of it as a way to shift the burden of tool definitions and execution away from your server and to MCP servers. Wihtout this you might have to write schema and fucntions for every functionality yoursel.

##### MCP CLient

The mcp client serves as a communication bridge between your server and MCP servers.

**Transport Agnostic Communication:**
One of MCP's key strengths is being transport agnostic - a fancy way of saying the client and server can talk to each other using different communication methods. The most common setup runs both the MCP client and server on the same machine, where they communicate through standard input/output.


We will uild a chatbot that allows users to interact with a collection of documents thorugh a comman-line interface. 


In [ ]:
## This intialises an MCP server

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")

In [2]:
docs = {
    "deposition.md": "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "financials.docx": "These financials outline the project's budget and expenditure",
    "outlook.pdf": "This document presents the projected future performance of the",
    "plan.md": "The plan outlines the steps for the project's implementation.",
    "spec.txt": "These specifications define the technical requirements for the equipment"
}

In [8]:
from pydantic import Field



@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    return docs[doc_id]

In [9]:
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the documents content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)

#### Server Inspector

When building MCP servers, you need a way to test your functionality without connecting to a full application. The python MCP SDK includes a build in browser based inspector that lets you debug and test the server in real time.

### Implementing a Client

In [13]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPClient:
    def __init__(self, command: str, args: list[str]):
        self._command = command
        self._args = args
        self._session = None
        self._ctx = None

    async def __aenter__(self):
        params = StdioServerParameters(command=self._command, args=self._args)
        self._ctx = stdio_client(params)
        read, write = await self._ctx.__aenter__()
        self._session_ctx = ClientSession(read, write)
        self._session = await self._session_ctx.__aenter__()
        await self._session.initialize()
        return self

    async def __aexit__(self, *exc):
        await self._session_ctx.__aexit__(*exc)
        await self._ctx.__aexit__(*exc)

    def session(self):
        if self._session is None:
            raise RuntimeError("Not connected - use 'async with'")
        return self._session

    async def list_tools(self):
        result = await mcp.call_tool(block.name, block.input)

In [16]:
@mcp.resource(
    "docs://documents",
    mime_type="application/json"
)
def list_docs() -> list[str]:
    return list(docs.keys())

In [17]:
@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]